# Decision Process Order-Driven

## To best decision system

## Name and Mean

|                  |main | 2nd | 
| ---------------- | --- | --- | 
| action           | A   |     | 
| State            | S   | S_1 | 
| target           | T   |     | 
| history          | H   |     | 
| compression unit | Z   |     | 
| candidates       | K   |     | 


In [13]:
# just import
import jax
import jax.numpy as jnp
import flax.linen as nn

In [14]:
def get_sinusoidal_positional_encoding(seq_len, d_model):
    """
    sin cos encoder
    """
    position = jnp.arange(seq_len)[:, None]
    div_term = jnp.exp(jnp.arange(0, d_model, 2) * (-jnp.log(10000.0) / d_model))
    
    pe_sin = jnp.sin(position * div_term)
    pe_cos = jnp.cos(position * div_term)
    
    pe = jnp.zeros((seq_len, d_model))
    pe = pe.at[:, 0::2].set(pe_sin)
    pe = pe.at[:, 1::2].set(pe_cos)
    return pe

In [ ]:
@jax.jit
def Y_attention_state_5_1_Bace(
    actions, target, state, histry,
    k_s, v_s,
    dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l
):
    """Simple and 5th idea bace
    """
    # 1. actions, goal, state concatate and attention
    q_ags = jnp.concatenate([actions, target, state, histry], axis=0)
    y_attention_state = nn.dot_product_attention(
        query=q_ags,
        key=k_s,
        value=v_s,
        bias=mask_s,
        dropout_rate=dropout_rate_s,
        deterministic=not dropout_enabled_s,
        dropout_rng=rng_s
    )


    return y_attention_state

In [ ]:
@jax.jit
def Y_attention_state_5_2(
    actions, target, state, histry,
    k_s, v_s,
    dropout_rate_s, dropout_enabled_s, rng_s, mask_s,
    w_dense, b_dense, num_l,
    mask_histry=None
):
    """
        idea5, type1 channel separation 
        A function that separates the attention of actions, 
        goals, and states from the self-attention of history, and integrates them using a fully connected layer.
    """
    # 1. actions, goal, state concatate and attention
    q_ags = jnp.concatenate([actions, target, state], axis=0)
    y_ags = nn.dot_product_attention(
        query=q_ags,
        key=k_s,
        value=v_s,
        bias=mask_s,
        dropout_rate=dropout_rate_s,
        deterministic=not dropout_enabled_s,
        dropout_rng=rng_s
    )

    # 2. attention to histry
    seq_len, d_model = histry.shape[-2], histry.shape[-1]
    pe = get_sinusoidal_positional_encoding(seq_len, d_model)
    histry_pe = histry + pe
    y_histry = nn.dot_product_attention(
        query=histry_pe,
        key=histry_pe,
        value=histry_pe,
        bias=mask_histry,
        dropout_rate=dropout_rate_s,
        deterministic=not dropout_enabled_s,
        dropout_rng=rng_s
    )

    # 3. liner 
    y_combined = jnp.concatenate([y_ags, y_histry], axis=0)

    y_attention_state = jnp.dot(y_combined, w_dense) + b_dense

    return y_attention_state

In [ ]:
@jax.jit
def Y_attention_state_5_3_L_1(actions, target, state, histry, dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, mask_histry=None):
    """ idea5, type2 channel separation.
        channel is A * T+S * S
        Enable Position encoding
    """
    def make_qkv(actions, target, state, histry):
        q_seq_len = actions.shape[-2] 
        q_d_model = actions.shape[-1]
        q_pe = get_sinusoidal_positional_encoding(q_seq_len, q_d_model)
        q = actions + q_pe

        k_bace = jnp.concatenate([target, state], axis=0)
        k_seq_len = k_bace.shape[-2]
        k_d_model = k_bace.shape[-1]
        k_pe = get_sinusoidal_positional_encoding(k_seq_len, k_d_model)
        k = k_bace + k_pe

        v_seq_len = histry.shape[-2]
        v_d_model = histry.shape[-1]
        v_pe = get_sinusoidal_positional_encoding(v_seq_len, v_d_model)
        v = histry + v_pe

        qkv = jnp.concatenate([q, k, v], axis=0)
        return (qkv, q_seq_len, k_seq_len, v_seq_len)

        
    def attention(carry, _):
        qkv = qkv_init + carry
        q, k, v = jnp.split(qkv, [q_split_idx, k_split_idx], axis=0)

        y_attention_state_carry_q = nn.dot_product_attention(
            query=q,
            key=q,
            value=q,
            bias=mask_s,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )
        y_attention_state_carry_k = nn.dot_product_attention(
            
            query=k,
            key=k,
            value=k,
            bias=mask_s,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )
        y_attention_state_carry_v = nn.dot_product_attention(
            query=v,
            key=v,
            value=v,
            bias=mask_histry,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )

        y_attention_state_carry = jnp.concatenate([y_attention_state_carry_q, y_attention_state_carry_k, y_attention_state_carry_v], axis=0)

        return y_attention_state_carry, None
    
    qkv_init, q_seq_len, k_seq_len, v_seq_len = make_qkv(actions, target, state, histry)
    q_split_idx = q_seq_len
    k_split_idx = q_seq_len + k_seq_len

    carry_init = jnp.zeros_like(qkv_init) 
    loop_trigger = jnp.arange(num_l)

    final_carry, _ = jax.lax.scan(attention, carry_init, loop_trigger)
    y_attention_state = qkv_init + final_carry

    return y_attention_state



In [18]:
@jax.jit
def Get_next_state(y_attention_state, state_index):
        """ This is Transformer thought best next state."""
        state_softmax = nn.softmax(y_attention_state[state_index])
        state_next = jnp.argmax(state_softmax)  
        return (state_next)

In [19]:
@jax.jit
def make_array_to_action(state, next_state, actions, p_coe, r_coe, gamma_coe):
    """
    S,S' = already,
    a = alredy,
    P = P(%)_coefficient * gradient of S -> S'
    R = R(%)_coefficient * gradient of S -> S'
    Gamma = G(%)_coefficient * gradient of S -> S' 
    How to set coefficient? I dont know :/
    """
    gradient = state/next_state * 100
    P = p_coe * gradient
    R = r_coe * gradient
    G = gamma_coe * gradient

    return (jnp.column_stack([state, next_state, actions, P, R, G]))

In [20]:
# Next, i think think next action to allive to state_next. and to cut cost, maybe MDP is better. but this is study so use transformer.
@jax.jit
def y_attention_action(array_to_action, k_a, v_a, dropout_rate_a, dropout_enabled_a, rng_a, mask_a):
    """This function return Next step **action**.
       but i think transformer is hevy. so just test
       array must include reword of action, % of go state when use actuion(i), gamma
       And, i think dropout must not use. It feels strange to train by logically dropping out groups of choices. I have no evidence for this.
    """
    q = array_to_action

    y_attention_action = nn.dot_product_attention( 
        query = q, 
        key = k_a, 
        value = v_a, 
        bias = mask_a,
        dropout_rate = dropout_rate_a,
        deterministic = not dropout_enabled_a,
        dropout_rng = rng_a
    )

    return (y_attention_action)

In [21]:
@jax.jit
def y_mdp_action(array_to_action):
    """
    This is simple **Bellman optimality equation**
    Latex 
    Q^*(s, a) = \sum_{s'} P(s' | s, a) \left[ R(s, a, s') + \gamma \max_{a'} Q^*(s', a') \right]
    """
    S = array_to_action[:,0]
    S_1 = array_to_action[:,1]
    A = array_to_action[:,2]
    P = array_to_action[:,3]
    R = array_to_action[:,4]
    G = array_to_action[:,5]

    Q = R * G * P * S_1


    

<>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/tmp/ipykernel_10/962918539.py:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  Q^*(s, a) = \sum_{s'} P(s' | s, a) \left[ R(s, a, s') + \gamma \max_{a'} Q^*(s', a') \right]
